# 🏋️‍♀️ 03: Curate & Extract Training Data

Join a run's trajectories and evaluations, apply a configurable quality
profile, and write the passing subset to
`output/curated/run=<id>/profile=<name>/`. This drives the shared `selection`
package, so `usersim select` runs the same code.

The curated output is the input to training-record shaping, not a replacement
for it: it carries the full trajectory schema plus `selection_*` quality
columns, so a downstream job can filter further without re-reading the raw
run.

## What the curated output can feed

Record shaping is left to you, because the thresholds and the review process
belong to whoever owns the resulting model. These are the shapes the schema
is designed to support, and the columns each one needs:

| Type | Source | Filter | Format |
|------|--------|--------|--------|
| **SFT** (supervised fine-tuning) | High-scoring assistant turns (incl. the `financial_services` **dynamic** tier: grounded, judge-scored answers) | `per_axis_overall_mean >= threshold` per axis × clean simulator status | `(prompt, response)` pairs |
| **Pairwise** (preference pairs) | Matched-pair runs against two assistant models (or two prompt variants) | Different scores on the same `persona_uuid + probe_variant + prompt_version` | `(prompt, chosen, rejected)` |
| **Verifier** | Trajectories with deterministic ground truth: tool-calling **and** the `financial_services` **verifiable** tier (`gold_tool_sequence` + `expected_state_deltas`) | `evaluator/scorers/tool_use` 8-axis scores; finance verifier pass/fail | `(state, action, verifier_label)` |
| **PRM** (process reward) | Per-turn judge outputs from `simulation_traces` | `assistant_quality` trace entries with sufficient n | `(prefix, step, step_score)` |
| **Online RL** | Branching counterfactuals (replay a turn under a different sampling) | Disagreement between branch outcomes | `(state, action_a, action_b, branch_outcomes)` |

## What it reads

Both produced by `01_simulate` and `02_evaluate_simulation`:

- `output/trajectories/` (Hive-partitioned by `locale=X/probe_family=Y/`): `simulation_outcome` (status, failure attribution, counters, per-model resource), `simulation_traces` (per-turn / per-call telemetry), `conversation_messages` (the trajectory itself), `persona_uuid`, `trajectory_id`.
- `output/evaluations/` (same Hive partitioning): wide `assistant_eval` cells (per-axis × per-judge scores + scorer outputs).

## What it writes

- `output/curated/run=<id>/profile=<name>/locale=X/probe_family=Y/part-*.parquet`: the passing trajectories (full schema + `selection_*` quality columns)
- `output/curated/run=<id>/profile=<name>/_selection_manifest.json`: the funnel, the profile, and provenance

Provenance carries `trajectory_id`, `persona_uuid`, `code_sha`, the model
identities and `prompt_version`, so a downstream training run can audit the
population it drew from and pin its inputs.

## Before you train on this

Two things are worth deciding deliberately rather than inheriting from the
defaults:

- **Acceptance thresholds are per-axis, per-probe and per-locale.** A single
  global cutoff will over-select the locales and probes that happen to score
  highest, which is a property of the scoring, not of the model.
- **Hold a slice out.** If every passing trajectory becomes training data,
  the evaluation and the training set share a population, and later
  measurements cannot tell improvement from memorization. The profile's
  holdout fraction exists for this.

In [ ]:
# Stage 1: Trajectory selection (load + select; preview & write come next).
#
# Drives the shared `selection` package (identical to `usersim select`):
#   load_runs -> select_trajectories -> [preview] -> write_curated_dataset -> [push_to_hf]
#
# Tune the profile knobs below; the GENEROUS default throws out only clear
# violations/errors. Re-run with a different profile name and the output is
# namespaced (`profile=<name>/`) so prior curations are never clobbered.

import dataclasses

from usersim.selection import load_runs, select_trajectories
from usersim.selection.profiles import available_profiles, get_profile

print("available profiles:", available_profiles())

# --- config -----------------------------------------------------------------
TRAJECTORIES = "../output/trajectories"
EVALUATIONS = "../output/evaluations"
OUT = "../output/curated"

# Pull from one or more runs. A list concatenates runs (deduped by
# trajectory_id) so you can top up a run that had failures with another.
RUNS = None   # None = the latest run; or a single id, or a list to concatenate

PROFILE = "generous"        # see available_profiles() for the shipped set
HOLDOUT_FRACTION = 0.0      # >0 reserves a deterministic persona slice as "holdout"
MIN_TURNS = None            # set an int to require multi-turn conversations

# Assistant language-match gate: fraction of ASSISTANT turns whose language
# lingua detects as the requested one. Adjust or disable the profile's floor.
#   0.5  = profile default for `generous` (recommended for clean training data:
#          drop a trajectory if the assistant answered in-language <50% of turns)
#   0.3  = lenient: keep partial-language trajectories (low-resource languages
#          where the model code-switches to English for technical terms)
#   None = drop this gate entirely: rely on in-sim native-script enforcement +
#          manual review. Useful for the India multi-language PREVIEW, where the
#          assistant's language adherence is itself the finding you want to keep,
#          not a filter (otherwise Tamil/Odia/etc. get thinned out).
# Only bites lingua-detectable locales; romanized (_Latn_IN) and lingua-less
# native locales (Odia/Kannada/Malayalam/Nepali/Marathi) don't carry this axis.
REQUESTED_LANGUAGE_MATCH_RATE = 0
MAX_RECORDS = None          # global cap to N highest-quality records (None = keep all)
MAX_RECORDS_PER_LOCALE = 10 # cap to N highest-quality passing records PER LOCALE
                            # (applied first; composes with MAX_RECORDS). None = no per-locale cap.
STRATIFY_BY = ["probe_family"]   # balance the GLOBAL cap across these cols; None = global top-K

# Exact columns written to parquet / uploaded to HF. Edit this list to write
# fewer or more: any name absent from the data is silently skipped, and
# locale/probe_family are always kept (the partitioned writer needs them).
# Run `result.curated.columns` (after the select cell) to see every column
# available to add (e.g. selection_quality_score, simulation_outcome, ...).
# Entries are resolved through `selection.resolve_columns`, so you can also
# drop in LOGICAL GROUP TOKENS that expand to several columns and are skipped
# where absent: e.g. "persona", "conversation", or "finance" (the
# financial_services gold-metadata flywheel signal: tier, gold_tool_sequence,
# expected_state_deltas, retrieved_document_ids, ...). Named schemas
# ("compact"/"full") are available the same way.
COLUMNS_TO_WRITE = [
    "run",
    "locale",
    "probe_family",
    "probe_type",
    "probe_variant",
    "theme",
    "num_turns",
    "num_tools",
    "tools",
    "trajectory_id",
    "persona_uuid",
    "persona_name",
    "persona_age",
    "persona_sex",
    "persona_education_level",
    "persona_occupation",
    "persona_location",
    # "persona_country",
    # "behavioral_profile",
    "disclosure_style",
    "user_interaction_style",
    # "conversation_language",
    "conversation_messages",
    # "conversation_metadata",
    # "conversation_status",
    # "finance",   # logical group: financial_services gold metadata (task_tier,
    #              # gold_tool_sequence, expected_state_deltas, retrieved_document_ids,
    #              # institution_type, dynamic_category_id, ...). Expands when present,
    #              # silently skipped for non-finance rows. Uncomment for finance runs.
]

# Generation LLM (assistant under test), recorded in the dataset card for
# provenance. It is not a trajectory column, but every run writes it to its
# _manifest.json, so the push cell below reads it from there. Leave this None
# unless you are curating runs whose manifest is missing or was produced
# elsewhere; a hand-entered value goes stale the first time you switch
# provider and forget to update it, and the card is a published artifact.
GENERATION_MODEL = None

profile = get_profile(PROFILE)
overrides = {}
if MIN_TURNS is not None:
    overrides["min_turns"] = int(MIN_TURNS)
if HOLDOUT_FRACTION:
    overrides["holdout_fraction"] = float(HOLDOUT_FRACTION)
if overrides:
    profile = dataclasses.replace(profile, **overrides)

# Adjust (or drop) the assistant language-match floor per the knob above.
# axis_floors is a frozen mapping, so build a new dict and replace.
_LANG_AXIS = "language.requested_language_match_rate"
if REQUESTED_LANGUAGE_MATCH_RATE is None:
    _floors = {k: v for k, v in profile.axis_floors.items() if k != _LANG_AXIS}
else:
    _floors = {**profile.axis_floors, _LANG_AXIS: float(REQUESTED_LANGUAGE_MATCH_RATE)}
if _floors != dict(profile.axis_floors):
    profile = dataclasses.replace(profile, axis_floors=_floors)
    print(
        f"language-match floor: "
        + ("dropped" if REQUESTED_LANGUAGE_MATCH_RATE is None
           else f"{REQUESTED_LANGUAGE_MATCH_RATE}")
        + f" (profile default was {get_profile(PROFILE).axis_floors.get(_LANG_AXIS)})"
    )

# --- load + select (no write yet) -------------------------------------------
traj_df, eval_df, run_id = load_runs(TRAJECTORIES, EVALUATIONS, runs=RUNS)
result = select_trajectories(
    traj_df, eval_df, profile=profile,
    max_records=MAX_RECORDS, max_per_locale=MAX_RECORDS_PER_LOCALE,
    stratify_by=STRATIFY_BY,
)

s = result.summary
_caps = []
if s.max_per_locale is not None:
    _caps.append(f"<={s.max_per_locale}/locale")
if s.max_records is not None:
    _caps.append(f"<={s.max_records} total")
print(f"runs {run_id} · profile {s.profile}")
print(f"  passed {s.passed}/{s.total}  (dropped {s.dropped}) · selected {s.selected}"
      + (f"  (caps: {', '.join(_caps)})" if _caps else ""))
for reason, count in sorted(s.drop_reasons.items(), key=lambda kv: (-kv[1], kv[0])):
    print(f"    drop: {reason:40s} {count}")
if s.holdout:
    print(f"  holdout split: {s.holdout}")

In [ ]:
# Preview EXACTLY what will be written/uploaded (the COLUMNS_TO_WRITE columns),
# before any parquet/HF write happens. Edit COLUMNS_TO_WRITE above and re-run.
import json

import pandas as pd

from usersim.selection import project

try:
    from IPython.display import display
except ImportError:  # running headless / as a script
    display = print

payload = project(result.curated, COLUMNS_TO_WRITE)
cols = list(payload.columns)
print(f"{len(cols)} columns, {len(payload)} rows")
print("columns:", cols)
dropped = [c for c in COLUMNS_TO_WRITE if c not in cols]
if dropped:
    print("requested but absent (skipped):", dropped)

# Where the kept records come from (per source run, per locale/probe_family).
if "run" in payload.columns:
    print("\nselected per source run:")
    for run, count in payload["run"].astype(str).value_counts().sort_index().items():
        print(f"  run {run:14s} {count}")
print("\nselected by locale/probe_family:")
for group, count in sorted(result.summary.kept_by_group.items()):
    print(f"  {group:40s} {count}")

# First rows of the projected payload (ALL columns that will be written;
# long cell values are truncated for readability, not dropped).
with pd.option_context(
    "display.max_columns", None,
    "display.width", None,
    "display.max_colwidth", 40,
):
    display(payload.head(5))

# A sample conversation from the highest-quality kept record. A run scored
# without judge axes has no quality score, so it shows the first kept record.
if len(result.curated):
    row = result.curated.sort_values("selection_quality_score", ascending=False).iloc[0]
    quality = row["selection_quality_score"]
    heading = "kept trajectory" if pd.isna(quality) else "top-quality kept trajectory"
    quality_text = "n/a" if pd.isna(quality) else f"{quality:.3f}"
    print(
        f"\n{heading} {row['trajectory_id']} "
        f"(quality={quality_text}, axes={row['selection_axis_scores']}):"
    )
    for msg in json.loads(row["conversation_messages"]):
        text = msg["content"].replace("\n", " ")
        print(f"  [{msg['role']:9s}] {text[:160]}")

In [ ]:
# Write the curated subset to parquet (projected to COLUMNS_TO_WRITE),
# namespaced by run set + profile. Run once the preview above looks right.
from usersim.selection import write_curated_dataset

out_root = write_curated_dataset(
    result.curated, OUT, run_id, profile,
    summary=result.summary, columns=COLUMNS_TO_WRITE,
)
print(f"wrote {result.summary.selected} records ({len(payload.columns)} cols) -> {out_root}")

In [ ]:
# Optional Hugging Face upload. Leave HF_REPO = None to skip, which is the
# default: nothing is uploaded unless you name a repo you own.
# Auth via HF_TOKEN env var or a cached `huggingface-cli login`.
HF_REPO = None          # e.g. "your-org/usersim-rlhf-en-us-1k"
HF_PRIVATE = True


# Hugging Face credentials (only prompts when HF_REPO is set). The token is a
# write-scoped HF access token (https://huggingface.co/settings/tokens). It is
# read here via getpass so it never gets baked into the notebook; falls back to
# the HF_TOKEN env var / a cached `huggingface-cli login` if you leave it blank.
import getpass
import os

HF_TOKEN = None
if HF_REPO:
    HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass(
        "Hugging Face write token (blank = use cached login): "
    ) or None
    print("HF token captured." if HF_TOKEN else "No token entered: using cached HF login.")
else:
    print("HF_REPO not set: skipping credential prompt.")

In [ ]:
# Optional: upload the curated subset to a (private) Hugging Face dataset repo.
# No-op unless HF_REPO is set above. Uploads the COLUMNS_TO_WRITE columns as
# a single parquet split + a dataset card + the selection manifest, using the
# token captured by the getpass cell above.
from usersim.selection import build_manifest, push_to_hf, resolve_generation_model
from usersim.selection.io import _provenance

if HF_REPO:
    # Read the model under test from each source run's manifest, so the card
    # cannot disagree with what actually generated the data. Names every
    # distinct model when concatenating runs.
    gen_model = resolve_generation_model(
        TRAJECTORIES, str(run_id), explicit=GENERATION_MODEL
    )
    print(f"generation model: {gen_model or 'unspecified (no manifest found)'}")
    manifest = build_manifest(
        profile=profile, summary=result.summary, source_run=str(run_id),
        eval_column="assistant_eval",
        provenance=_provenance(result.curated, "assistant_eval"),
        generation_model=gen_model,
    )
    url = push_to_hf(
        result.curated, HF_REPO, private=HF_PRIVATE, token=HF_TOKEN,
        manifest=manifest, profile=profile, columns=COLUMNS_TO_WRITE,
    )
    print(f"uploaded {len(result.curated)} rows -> {url} (private={HF_PRIVATE})")
else:
    print("HF_REPO not set: skipping upload.")